In [ ]:
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

def train_test_anomaly_detection(train_csv_path, test_csv_path, model_type='ocsvm', model_params=None):
    # Load and preprocess data
    train_data = pd.read_csv(train_csv_path)
    test_data = pd.read_csv(test_csv_path)
    train_data.fillna(0, inplace=True)
    test_data.fillna(0, inplace=True)

    # Extract features and labels from test data
    X_train = train_data.drop(columns=['Label'])
    X_test = test_data.drop(columns=['Label'])
    y_test = test_data['Label']

    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Initialize the model
    if model_type == 'ocsvm':
        model = OneClassSVM(**(model_params or {}))
    elif model_type == 'iforest':
        model = IsolationForest(**(model_params or {}))
    elif model_type == 'lof':
        model = LocalOutlierFactor(**(model_params or {}))
    elif model_type == 'kmeans':
        model = KMeans(**(model_params or {}))
    elif model_type == 'dbscan':
        model = DBSCAN(**(model_params or {}))
    else:
        raise ValueError("Unsupported model type")

    # Train and predict
    if model_type in ['ocsvm', 'iforest', 'kmeans', 'dbscan']:
        model.fit(X_train)
        y_pred_test = model.predict(X_test)
        if model_type != 'kmeans':
            y_pred_test = [1 if pred == -1 else 0 for pred in y_pred_test]
    elif model_type == 'lof':
        model.fit(X_train)
        y_pred_test = model.fit_predict(X_test)
        y_pred_test = [1 if pred == -1 else 0 for pred in y_pred_test]

    # Handle KMeans specific case
    if model_type == 'kmeans':
        distances = model.transform(X_test)
        y_pred_test = [1 if min(dist) > some_threshold_value else 0 for dist in distances]

    # Evaluate the model
    print("Classification Report:\n", classification_report(y_test, y_pred_test))
    print("Accuracy Score:", accuracy_score(y_test, y_pred_test))

    # Confusion matrix for additional metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    fpr = fp / (fp + tn)
    tpr = tp / (tp + fn)

    print("False Positive Rate:", fpr)
    print("True Positive Rate:", tpr)

# Example usage
train_test_anomaly_detection(
    '../data_selected/syncan/train.csv', 
    '../data_selected/syncan/test_flooding.csv', 
    model_type='ocsvm', 
    model_params={'gamma': 'auto', 'nu': 0.05})


In [ ]:
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

def train_test_anomaly_detection(train_csv_path, test_csv_path, model_type='ocsvm', model_params=None):
    tqdm.write("Loading and preprocessing data...")
    # Load and preprocess data
    train_data = pd.read_csv(train_csv_path)
    test_data = pd.read_csv(test_csv_path)
    train_data.fillna(0, inplace=True)
    test_data.fillna(0, inplace=True)

    # Extract features and labels from test data
    X_train = train_data.drop(columns=['Label'])
    X_test = test_data.drop(columns=['Label'])
    y_test = test_data['Label']

    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    tqdm.write("Initializing and training the model...")
    # Initialize the model
    if model_type == 'ocsvm':
        model = OneClassSVM(**(model_params or {}))
    elif model_type == 'iforest':
        model = IsolationForest(**(model_params or {}))
    elif model_type == 'lof':
        model = LocalOutlierFactor(**(model_params or {}))
    elif model_type == 'kmeans':
        model = KMeans(**(model_params or {}))
    elif model_type == 'dbscan':
        model = DBSCAN(**(model_params or {}))
    else:
        raise ValueError("Unsupported model type")

    # Train and predict
    if model_type in ['ocsvm', 'iforest', 'kmeans', 'dbscan']:
        model.fit(X_train)
        y_pred_test = model.predict(X_test)
        if model_type != 'kmeans':
            y_pred_test = [1 if pred == -1 else 0 for pred in y_pred_test]
    elif model_type == 'lof':
        model.fit(X_train)
        y_pred_test = model.fit_predict(X_test)
        y_pred_test = [1 if pred == -1 else 0 for pred in y_pred_test]

    # Handle KMeans specific case
    if model_type == 'kmeans':
        tqdm.write("Handling KMeans specific case...")
        distances = model.transform(X_test)
        y_pred_test = [1 if min(dist) > some_threshold_value else 0 for dist in distances]

    tqdm.write("Evaluating the model...")
    # Evaluate the model
    print("Classification Report:\n", classification_report(y_test, y_pred_test))
    print("Accuracy Score:", accuracy_score(y_test, y_pred_test))

    # Confusion matrix for additional metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()
    fpr = fp / (fp + tn)
    tpr = tp / (tp + fn)

    print("False Positive Rate:", fpr)
    print("True Positive Rate:", tpr)

    tqdm.write("Process completed.")

# Example usage
train_test_anomaly_detection(
    '../data_selected/syncan/train.csv', 
    '../data_selected/syncan/test_flooding.csv', 
    model_type='ocsvm', 
    model_params={'gamma': 'auto', 'nu': 0.05})
